# Quantization: fixed calibration, clipping, and error

Prerequisites: floating-point representation and rounding. Estimated time: 30 minutes. Objectives: quantize with a fixed calibration scale, expose clipping, compute reconstruction error, and compare ideal storage.

This standard-library example quantizes toy weights to a signed five-level codebook. It compares fp32 storage with idealized 3-bit storage and measures reconstruction error. No model or hardware kernel is loaded.

## Baseline and calibration assumption

Use symmetric codes -2 through 2 and a fixed scale of 0.5 chosen from a separate calibration range. This is a five-level codebook with representable dequantized range [-1, 1]. The evaluation weights include an outlier at 1.5 to expose clipping.

In [ ]:
weights = [-1.0, -0.5, 0.0, 0.5, 1.0, 1.5]
scale = 0.5
qmin, qmax = -2, 2
baseline_bytes = len(weights) * 4
assert baseline_bytes == 24
print('fp32 reference values:', weights, 'ideal bytes:', baseline_bytes)

## Quantize and dequantize

Quantize with round(x / scale), clip to the code range, then dequantize as q * scale. In this toy, ties use Python's round-to-even rule; production formats specify their rounding behavior explicitly.

In [ ]:
def quantize(value, scale, qmin, qmax):
    raw = round(value / scale)
    code = min(qmax, max(qmin, raw))
    return code, code * scale, code != raw
results = [quantize(w, scale, qmin, qmax) for w in weights]
codes = [row[0] for row in results]
reconstructed = [row[1] for row in results]
clipped = [weights[i] for i, row in enumerate(results) if row[2]]
assert codes == [-2, -1, 0, 1, 2, 2]
assert reconstructed == [-1.0, -0.5, 0.0, 0.5, 1.0, 1.0]
assert clipped == [1.5]
print('codes:', codes, 'reconstructed:', reconstructed, 'clipped inputs:', clipped)

## Comparison and metric

Mean squared reconstruction error compares the toy quantized values to the reference. Five distinct codes require at least three bits each. The ideal bit count is 18 bits for six values, or 2.25 byte-equivalents versus 24 fp32 bytes (10.67x smaller). A byte-aligned packed buffer needs at least 3 bytes, before scales, alignment, metadata, and runtime overhead. This ratio does not predict latency or model quality.

In [ ]:
mse = sum((a-b)**2 for a,b in zip(weights, reconstructed)) / len(weights)
quantized_bytes_ideal = len(weights) * 3 / 8
compression_ratio = baseline_bytes / quantized_bytes_ideal
assert round(mse, 6) == 0.041667
assert round(compression_ratio, 2) == 10.67
print('MSE:', round(mse, 6), 'ideal storage ratio:', compression_ratio, 'x')

## Edge and failure cases

A calibration scale that is too small clips many values; one that is too large wastes codes and increases rounding error near zero. Empty inputs need an explicit metric policy. Real quantization evaluation also needs task slices and target-hardware latency/memory measurements.

In [ ]:
tiny_scale_results = [quantize(w, 0.25, qmin, qmax) for w in weights]
tiny_scale_clips = sum(row[2] for row in tiny_scale_results)
assert tiny_scale_clips >= 2
def mean_squared_error(reference, approx):
    if len(reference) != len(approx) or not reference:
        raise ValueError('equal non-empty arrays required')
    return sum((a-b)**2 for a,b in zip(reference, approx)) / len(reference)
try:
    mean_squared_error([], [])
except ValueError:
    pass
else:
    raise AssertionError('empty metric input must be rejected')

## Exercises

1. Recalibrate using maximum absolute value and compare clipping and MSE. Explain why that can hide outliers in a larger tensor.
2. Try per-group scales and report the additional scale metadata.
3. Propose a fixed evaluation set for a quantized language model, including rare and long-context cases.

## Summary

A fixed calibration scale defines representable values; clipping and rounding both contribute error. Report reconstruction and task metrics alongside actual packed size and target-runtime performance. See the [concept note](../concepts/30-quantization.md) and the [practice assessment](../interview-prep/curriculum-practice.md#quantization-for-llms).